#<img src="https://img.stackshare.io/service/10345/ADB.png"/></a> Workshop Lab 3 - Governance/Warehouse/AI/ETC#
**Content**
  * Unity Catalog
  * Workflow
  * Genie AI assist
  * SQL Warehouse
  * Other Data Platform
  

# Data processing framework

* Agenda
* Key termonologies
* GUI check
* MiniUsecase - Medallion with Lakeflow
    * create pipeline (Consists: 4x pipeline files, 1x notebook cell for data generation)
    * run, check the results
    * rerun
* Playing with Genie and SQL Editor



In [0]:
Lakehouse

# 🧭 ETL in Databricks

---

### **Core Concepts**

**Orchestration**  
The coordination and scheduling of multiple data processing tasks or workflows to ensure they run in the correct order and timing.

**Job**  
A single, executable unit of work in Databricks (e.g., running a notebook, script, or pipeline) that can be scheduled or triggered.

**Pipeline**  
A defined sequence of data transformations and movement steps (e.g., ETL/ELT) that process data from source to target, often managed and executed through orchestration.

---

### **Declarative Frameworks**

**LDP Lakeflow (Spark) Declarative Pipelines DLT ←→ Delta Live Tables  **

**Delta Live Tables (DLT) - kind of legacy**  
*A declarative framework for building reliable, maintainable, and testable data processing pipelines.*

**Lakeflow Declarative Pipelines (LDP)**  
*A declarative framework for developing and running batch and streaming data pipelines in SQL and Python.*

---

### **Approach Styles**

**Procedural approach**  
You define *how* tasks should run by specifying the exact sequence of operations and control flow.

**Declarative approach**  
You describe *what* result you want, and the system figures out how to execute it optimally.

---

### **Pipeline objects**

**Sink**  
A target destination (e.g., Delta table, Kafka topic, Event Hub) for streaming flows where processed records are written out via a pipeline.

**Streaming table**  
A Unity Catalog–managed table defined for incremental or streaming ingestion and processing, backed by a serverless pipeline, and refreshed on a schedule or trigger.  

**Materialized view**  
A persistent table that stores the precomputed results of a query, updates incrementally or on schedule, and provides faster query performance than a standard view.

**Flow**  
A named data path that reads from a source, applies processing logic, and writes results into a dataset. Every streaming table and materialized view has one default flow (named after its target); extra **append flows** can write into the same streaming table (fan-in, new sources, backfills).


<div align="center">
  <img src="https://docs.databricks.com/aws/en/assets/images/dlt-core-concepts-6bc9894d3682035cadac19f3980875ae.png" width="800"/>
</div>

<div align="center">
<img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*MggtENQXxtieCKA30b4xbg.png" width="800"/>
</div>

#Mini Usecase
![image_1777449282252.png](../images/image_1777449282252.png "image_1777449282252.png")
## Steps:
  * Emulate the data source with Notebook cell
  * Create medallion layers via Lakeflow Pipelines (old terminology: DLT) (bronze, silver, gold):
  * Run it and test


In [0]:
# Idempotent: safe to run anytime (re-creates the volume and folder if missing)
# Create the source layer volume and folder
catalog = "workspace"
schema = "default"
volume_folder = f"/Volumes/{catalog}/{schema}/raw_data"

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`raw_data`")

def ensure_empty_folder(path):
    # Remove any existing contents (no-op if the folder doesn't exist yet)
    dbutils.fs.rm(path, True)
    # Create the folder and any missing parents
    dbutils.fs.mkdirs(path)


ensure_empty_folder(f"/Volumes/{catalog}/{schema}/raw_data/customers")


In [0]:
#Check
dbutils.fs.ls(f"/Volumes/{catalog}/{schema}/raw_data/customers")

In [0]:
#Step 0 - Generate some source data
%pip install faker
from pyspark.sql import functions as F
from faker import Faker
from collections import OrderedDict
import uuid
import random

#Configure fields
fake = Faker()
# docs: https://github.com/xfxf/faker-python

fake_firstname = F.udf(lambda: fake.first_name())
fake_lastname = F.udf(lambda: fake.last_name())
fake_email = F.udf(lambda: fake.ascii_company_email())
#fake_birth_date = F.udf(lambda:fake.date_time_this_century().strftime("%m-%d-%Y"))
fake_birth_date = F.udf(lambda:fake.date_between_dates(date_start="-90y", date_end="-2y").strftime("%m-%d-%Y"))
# Limited country list for cleaner grouping/analysis
available_eu_countries = [
    "Hungary", "Germany", "Austria", "Slovakia", "Poland",
    "Czech Republic", "Romania", "France", "Italy", "Spain",
    "Netherlands", "Slovenia", "Belgium", "Greece", "Portugal", "Sweden", "Norway", "Denmark", "Finland", "Ireland", "United Kingdom", "Luxembourg", "Malta", "Iceland", "Liechtenstein", "Andorra", "Monaco", "San Marino", "Vatican City", "Bulgaria", "Cyprus", "Estonia", "Latvia", "Lithuania", "Albania", "Bosnia and Herzegovina"
]
fake_country = F.udf(lambda: random.choice(available_eu_countries))
fake_registration_date = F.udf(lambda:fake.date_time_this_decade().strftime("%m-%d-%Y %H:%M:%S"))
fake_id = F.udf(lambda: str(uuid.uuid4()) if random.uniform(0, 1) < 0.98 else None)
categories = OrderedDict([("RED", 0.5),("GREEN", 0.1),("BLUE", 0.3),(None, 0.01)])
fake_categories = F.udf(lambda:fake.random_elements(elements=categories, length=1)[0])

# Set the record number
number_of_customer = 99
#optional performance step
df = spark.range(0, number_of_customer).repartition(2)

df = df.withColumn("id", fake_id())
df = df.withColumn("firstname", fake_firstname())
df = df.withColumn("lastname", fake_lastname())
df = df.withColumn("date_of_birth", fake_birth_date())
df = df.withColumn("email", fake_email())
df = df.withColumn("country_of_origin", fake_country())
df = df.withColumn("customer_category", fake_categories())
df = df.withColumn("registration_date", fake_registration_date())
df_customers = df.withColumn("load_ts", F.current_timestamp())

print(df_customers.count())
df_customers.repartition(20).write.format("json").mode("append").save(volume_folder+"/customers")

print(f"{number_of_customer} customers are generated. Process done..")

In [0]:
# Challenge
# Check the data

df = spark.read.format("json").load(volume_folder+"/customers")
#df.groupby("country_of_origin").count().orderBy("count").display()
df.limit(10).display()

In [0]:
# Clean up
# Remove the generated data
dbutils.fs.rm(volume_folder + "/customers", True)

# Drop the volume (also deletes all files stored in it)
spark.sql(f"DROP VOLUME IF EXISTS `{catalog}`.`{schema}`.`raw_data`")

spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_customers")
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_customers")
spark.sql("DROP TABLE IF EXISTS workspace.default.customers_categories") 
spark.sql("DROP TABLE IF EXISTS workspace.default.customers_registration_monthly") 
spark.sql("DROP TABLE IF EXISTS workspace.default.customers_by_country") 




### Source data: `customers` (JSON)

Each run of **Step 0** appends `number_of_customer` (default 1000) fake customer records as JSON files under `{volume_folder}/customers`. Some data-quality quirks are added on purpose: ~2% of rows have a `null` id, some rows have a `null` `customer_category`, and dates are stored as strings rather than typed values.

| Field | Description |
| --- | --- |
| `id` | Unique customer identifier (UUID string). ~2% of rows are `null` to simulate records with a missing key. |
| `firstname` | Fake first name (Faker). |
| `lastname` | Fake last name (Faker). |
| `date_of_birth` | Fake date within the current century, stored as string in `MM-DD-YYYY` format (not a date type). |
| `email` | Fake company email address (Faker `ascii_company_email`). |
| `country_of_origin` | One of 36 fixed European countries (random pick from a limited list), so country grouping stays clean. |
| `customer_category` | Weighted random category: `RED` (~55%), `BLUE` (~33%), `GREEN` (~11%), `null` (~1%) — tests null handling and filtering downstream. |
| `registration_date` | Fake datetime within the current year, stored as string in `MM-DD-YYYY HH:MM:SS` format (not a timestamp type). |
| `load_ts` | Timestamp of record generation (`current_timestamp()`), typed as timestamp — usable as the batch/ingestion marker. |

#Q&A


contacts: tamas.papp@t-systems.com, 
         agnes.gerse@t-systems.com

Docs:
- https://docs.databricks.com/aws/en/getting-started/data-pipeline-get-started
- https://youtu.be/krzr52wd4nM
- https://docs.databricks.com/aws/en/data-engineering/#gsc.tab=
- https://docs.databricks.com/aws/en/assets/images/dlt-core-concepts-6bc9894d3682035cadac19f3980875ae.png
- https://docs.databricks.com/aws/en/ldp/concepts#gsc.tab=0
- https://docs.databricks.com/aws/en/ldp/develop#gsc.tab=0